In [10]:
from langchain_community.document_loaders import DirectoryLoader, PyMuPDFLoader
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_openai import ChatOpenAI
from langchain_tavily import TavilySearch
from langgraph.graph import StateGraph, END
from typing import TypedDict
from dotenv import load_dotenv
import os

### Building Components

In [11]:
def doc_loader(path):
    try:
        loader = DirectoryLoader(path,
                                glob="**/*.pdf",
                                loader_cls=PyMuPDFLoader,
                                show_progress=True)
        documents = loader.load()
        print(f"Loaded {len(documents)} documents from {path}")
        return documents
    except Exception as e:
        print(f"Error loading documents from {path}: {e}")
        return None

def text_splitter(documents):
    print("Splitting documents into chunks...")
    try:
        if not documents:
            raise ValueError("No documents to split")
        splitter = RecursiveCharacterTextSplitter(chunk_size=1000, 
                                              chunk_overlap=150,
                                              length_function=len,
                                              separators=["\n\n", "\n", " ", ""])
        chunks = splitter.split_documents(documents)
        print(f"""Split into {len(chunks)} chunks
            Document splitting complete""")
        return chunks
    except Exception as e:
        print(f"Error splitting documents: {e}")
        return None

def create_vector(chunks):
    print('Loading embedding model...')
    embedding = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2"
        )
    print('Creating vector store...')
    vector_store = Chroma.from_documents(chunks, 
                                         embedding, 
                                         collection_name="pdf_docs")
    print('Vector store created successfully')
    return vector_store

def llm_model():
    print('Loading LLM model...')
    llm = ChatOpenAI(
        model="moonshotai/kimi-k2.6:free", 
        api_key=os.getenv("OPENROUTER_API_KEY"),
        base_url="https://openrouter.ai/api/v1")
    return llm

def web_search(state):
    search_tool = TavilySearch(
        max_results=3
    )

In [12]:
documents = doc_loader("../data")
chunks = text_splitter(documents)
vector_db = create_vector(chunks)

  0%|          | 0/3 [00:00<?, ?it/s]

100%|██████████| 3/3 [00:01<00:00,  2.39it/s]


Loaded 781 documents from ../data
Splitting documents into chunks...
Split into 1573 chunks
            Document splitting complete
Loading embedding model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Creating vector store...
Vector store created successfully


### using the above components to build Agentic RAG pipeline

In [13]:
# defining graph state
class agentstate(TypedDict):
    question:str
    rewritten_query:str
    documents:list
    web_result:str
    answer:str

### Re-writing the user query  

In [14]:
def rewrite(state):
    query = state["question"]
    llm = llm_model()
    prompt = f"""
    You are a query rewriting assistant.

    Rewrite the query only if it improves retrieval quality.

    Rules:
    - Preserve meaning
    - Do not add information
    - Return the original query if already clear

    Query:
    {query}
    """
    rewritten_query = llm.invoke(prompt)
    return {
        "rewritten_query":rewritten_query.content
    }


### Retrival Block

In [15]:
def retriever(state):
    query = state["rewritten_query"]
    retriever_db = vector_db.as_retriever()
    r_docs = retriever_db.invoke(query) 
    return {
        "documents":r_docs
    }

### Grade Retrived documents


In [16]:
def search_web(state):
    query = state["rewritten_query"]
    search_tool = web_search()
    results = search_tool.invoke(query)

    return {
        "web_result":results
    }

def grade_docuemnts(state):
    docs = state["documents"]
    if len(docs) > 0:
        return "Generate"
    else :
        return "web-search"

In [17]:
def generator(state):
    query = state["question"]
    docs = state.get("documents",[])
    web_results = state.get("web_results","")

    if docs:
        context = "\n\n".join(
            [doc.page_content for doc in docs]
        )
    else:
        context = web_results

    prompt = f"""
Answer the following using the provided context

question:{query}

context:{context}
"""
    llm = llm_model()
    response = llm.invoke(prompt)

    return {
        "answer":response.content
    }

In [18]:
workflow = StateGraph(agentstate)

workflow.add_node(
    "rewrite",
    rewrite
)

workflow.add_node(
    "retriever",
    retriever
)

workflow.add_node(
    "web_search",
    web_search
)

workflow.add_node(
    "generate",
    generator
)

In [19]:
workflow.set_entry_point("rewrite")

workflow.add_edge(
    "rewrite",
    "retriever"
)

workflow.add_conditional_edges(
    "retriever",
    grade_docuemnts,
    {
        "generate":"generate",
        "web_search":"web_search"
    }
)

workflow.add_edge(
    "web_search",
    "generate"
)

workflow.add_edge(
    "generate",
    END
)

graph = workflow.compile()

In [ ]:
user_query=input("enter a query:")
response = graph.invoke(
    {
        "question":user_query
    }
)

print(response["answer"])